# Problem 2 — Linear Algebra: Indirect Ownership and PCA
**Candidate:** Chetsadaphon Kanthiya 
**Status:** Compulsory (25 Points) 

---

## Part A — Indirect Ownership

### Problem Setup
Let $M$ be the direct-ownership matrix of size $4 \times 4$ for companies $A, B, C$, and $D$, where $M_{ij}$ represents the fraction of company $j$ held directly by company $i$:

$$
M = \begin{pmatrix}
0 & 0.6 & 0 & 0.1 \\
0 & 0 & 0.5 & 0 \\
0.2 & 0 & 0 & 0.4 \\
0 & 0 & 0 & 0
\end{pmatrix}
$$

The total ownership matrix (direct + indirect through every possible chain of length $k$) is defined as the infinite matrix series:
$$T = M + M^2 + M^3 + \dots = \sum_{k=1}^{\infty} M^k$$

### Question 1 (20%): Proof of Series Convergence & Spectral Radius Condition

#### 1. Mathematical Derivation of $T = (I - M)^{-1} - I$
Consider the partial sum of the Neumann series up to power $N$:
$$S_N = I + M + M^2 + M^3 + \dots + M^N = \sum_{k=0}^N M^k$$

Post-multiplying both sides by $M$:
$$S_N M = M + M^2 + M^3 + \dots + M^{N+1}$$

Subtracting the two equations:
$$S_N - S_N M = I - M^{N+1} \implies S_N (I - M) = I - M^{N+1}$$

For a finite matrix, the Neumann series converges if and only if $\rho(M) < 1$. Under this condition, $M^{N+1} \to \mathbf{0}$ and the partial sums $S_N$ converge. Taking the limit yields:
$$S (I - M) = I$$

Since $(I - M)$ is invertible under this condition, multiplying by $(I - M)^{-1}$ gives:
$$S = \sum_{k=0}^{\infty} M^k = (I - M)^{-1}$$

Subtracting the identity matrix $I$ (which represents baseline self-identity, not ownership):
$$T = \sum_{k=1}^{\infty} M^k = S - I = \mathbf{(I - M)^{-1} - I} \quad \blacksquare$$

#### 2. Convergence Condition in Terms of Spectral Radius
The matrix geometric series $\sum_{k=0}^{\infty} M^k$ converges if and only if:
$$\mathbf{\rho(M) < 1}$$
where $\rho(M)$ is the **spectral radius** of $M$, defined as the maximum absolute eigenvalue:
$$\rho(M) = \max_i |\lambda_i(M)|$$

#### 3. Why Real-World Ownership Matrices Satisfy this Condition
For an ownership matrix whose entries are non-negative fractions, each column sum is at most one when the represented holders collectively own no more than 100% of each company. Hence $\rho(M) \le \|M\|_1 = \max_j \sum_i M_{ij} \le 1$. This bound alone does not guarantee strict convergence: a closed ownership component with no leakage can have spectral radius 1. The series converges when every closed strongly connected ownership component has leakage to shareholders outside that component (so repeated ownership paths lose weight). External shareholders and public float often provide such leakage, but the condition must be checked rather than assumed.

### Question 2 (27%): Computing $T$ by Hand & with NumPy

#### 1. Hand Calculation using Block Upper Triangular Inversion
Notice that company $D$ (Row 4) holds 0% shares in any company ($M_{4, :} = [0, 0, 0, 0]$). This reveals a **Block Upper Triangular** structure:

$$M = \begin{pmatrix} M_{11} & M_{12} \\ \mathbf{0} & 0 \end{pmatrix}$$
where:
$$M_{11} = \begin{pmatrix} 0 & 0.6 & 0 \\ 0 & 0 & 0.5 \\ 0.2 & 0 & 0 \end{pmatrix}, \quad M_{12} = \begin{pmatrix} 0.1 \\ 0 \\ 0.4 \end{pmatrix}$$

Therefore, $(I - M)$ is also block upper triangular:
$$I - M = \begin{pmatrix} I_3 - M_{11} & -M_{12} \\ \mathbf{0} & 1 \end{pmatrix}$$

The block inverse formula gives:
$$(I - M)^{-1} = \begin{pmatrix} (I_3 - M_{11})^{-1} & (I_3 - M_{11})^{-1} M_{12} \\ \mathbf{0} & 1 \end{pmatrix}$$
This reduces our $4 \times 4$ matrix inversion problem down to a simple $3 \times 3$ inversion!

**Step A: Inverting $(I_3 - M_{11})$:**
$$I_3 - M_{11} = \begin{pmatrix} 1 & -0.6 & 0 \\ 0 & 1 & -0.5 \\ -0.2 & 0 & 1 \end{pmatrix}$$

- Determinant: $\det(I_3 - M_{11}) = 1(1 - 0) - (-0.6)(0 - 0.1) = 1 - 0.06 = \mathbf{0.94}$
 *(Notice $0.06 = 0.6 \times 0.5 \times 0.2$ is the product of weights around the circular loop $A \to B \to C \to A$)*

- Adjugate Matrix: Transpose of cofactors:
$$\text{adj}(I_3 - M_{11}) = \begin{pmatrix} 1 & 0.6 & 0.3 \\ 0.1 & 1 & 0.5 \\ 0.2 & 0.12 & 1 \end{pmatrix}$$

- Inverse:
$$(I_3 - M_{11})^{-1} = \frac{1}{0.94} \begin{pmatrix} 1 & 0.6 & 0.3 \\ 0.1 & 1 & 0.5 \\ 0.2 & 0.12 & 1 \end{pmatrix}$$

**Step B: Computing the 4th Column (Company D):**
$$(I_3 - M_{11})^{-1} M_{12} = \frac{1}{0.94} \begin{pmatrix} 1 & 0.6 & 0.3 \\ 0.1 & 1 & 0.5 \\ 0.2 & 0.12 & 1 \end{pmatrix} \begin{pmatrix} 0.1 \\ 0 \\ 0.4 \end{pmatrix} = \frac{1}{0.94} \begin{pmatrix} 0.1 + 0.12 \\ 0.01 + 0.20 \\ 0.02 + 0.40 \end{pmatrix} = \frac{1}{0.94} \begin{pmatrix} 0.22 \\ 0.21 \\ 0.42 \end{pmatrix}$$

**Step C: Final Calculation of $T = (I - M)^{-1} - I$:**
$$T = \frac{1}{0.94} \begin{pmatrix}
0.06 & 0.60 & 0.30 & 0.22 \\
0.10 & 0.06 & 0.50 & 0.21 \\
0.20 & 0.12 & 0.06 & 0.42 \\
0 & 0 & 0 & 0
\end{pmatrix}$$

#### Total Percentage Ownership of Company A in Company D:
$$T_{AD} = \frac{0.22}{0.94} = \frac{11}{47} \approx \mathbf{0.23404 \implies 23.40\%}$$

**Path Verification:** Direct path $A \to D = 0.10$. Indirect path $A \to B \to C \to D = 0.6 \times 0.5 \times 0.4 = 0.12$. Initial sum $= 0.10 + 0.12 = 0.22$. Factoring the infinite cycle multiplier $\frac{1}{1 - 0.06} = \frac{1}{0.94}$, total $= \frac{0.22}{0.94} = 23.40\%$.

In [5]:
# คำนวณเมทริกซ์การถือหุ้นรวมด้วย NumPy แล้วตรวจค่า spectral radius
# NumPy Implementation & Verification
import numpy as np
import pandas as pd

companies = ['A', 'B', 'C', 'D']
M = np.array([
 [0.0, 0.6, 0.0, 0.1],
 [0.0, 0.0, 0.5, 0.0],
 [0.2, 0.0, 0.0, 0.4],
 [0.0, 0.0, 0.0, 0.0]
])

# 1. Verify Spectral Radius
eigenvalues = np.linalg.eigvals(M)
spectral_radius = np.max(np.abs(eigenvalues))
print(f"Spectral radius rho(M) = {spectral_radius:.4f} < 1 (Convergence Guaranteed!)")

# 2. Calculate Total Ownership Matrix T
I = np.eye(4)
inv_I_M = np.linalg.inv(I - M)
T = inv_I_M - I

df_T = pd.DataFrame(T, index=companies, columns=companies)
print("\nTotal Ownership Matrix T (NumPy Result):")
print(df_T.round(4))

print(f"\nExact Total Ownership of Company A in Company D: {T[0, 3]:.4f} ({T[0, 3]*100:.2f}%)")

Spectral radius rho(M) = 0.3915 < 1 (Convergence Guaranteed!)

Total Ownership Matrix T (NumPy Result):
        A       B       C       D
A  0.0638  0.6383  0.3191  0.2340
B  0.1064  0.0638  0.5319  0.2234
C  0.2128  0.1277  0.0638  0.4468
D  0.0000  0.0000  0.0000  0.0000

Exact Total Ownership of Company A in Company D: 0.2340 (23.40%)


### Question 3 (13%): Business Meaning of Non-Zero Diagonal Entries & Stakeholder Reporting

#### 1. Business Meaning
The diagonal entries represent indirect self-ownership:
$$T_{AA} = T_{BB} = T_{CC} = \frac{0.06}{0.94} \approx \mathbf{6.38\%}$$
This means company $A$ indirectly holds $6.38\%$ of its own shares through the ownership loop $A \to B \to C \to A$. Companies $B$ and $C$ similarly hold $6.38\%$ in themselves.

#### 2. Stakeholder Reporting Protocol
1. **Disclose the circular path:** Report the $6.38\%$ as path-based indirect self-ownership implied by this model, and identify the A-B-C-A loop. It is not necessarily the same as shares directly held in treasury.
2. **Separate ownership from voting control:** Do not automatically subtract $6.38\%$ from voting shares. Effective voting rights and consolidation treatment depend on the legal entities, control relationships, and applicable accounting and securities rules.
3. **Request specialist review:** Show the direct ownership links and calculated indirect interests, explain the assumptions, and have legal/accounting specialists determine how circular interests affect voting, dividends, and consolidated reporting.

---

## Part B — PCA and SVD

### Question 1 (20%): Proof of Variance Maximization via Lagrange Multipliers

Let $X \in \mathbb{R}^{n \times p}$ be a centred data matrix with sample covariance matrix:
$$S = \frac{1}{n - 1} X^T X$$
where $S$ is symmetric and positive semi-definite ($S = S^T, S \succeq 0$).

#### 1. Projection and Variance
Projecting data onto a unit direction vector $w \in \mathbb{R}^p$ (subject to $w^T w = 1$) yields $z = Xw$.
Since $\mathbb{E}[X] = \mathbf{0}$, the variance of the projected data is:
$$\text{Var}(Xw) = \frac{1}{n - 1} (Xw)^T (Xw) = w^T \left(\frac{1}{n - 1} X^T X\right) w = \mathbf{w^T S w}$$

#### 2. Constrained Optimization Formulation
$$\max_w \quad w^T S w \quad \text{subject to } \quad w^T w = 1$$

Construct the Lagrangian function with multiplier $\lambda \in \mathbb{R}$:
$$\mathcal{L}(w, \lambda) = w^T S w - \lambda (w^T w - 1)$$

#### 3. First-Order Necessary Condition
Taking the gradient with respect to $w$:
$$\nabla_w \mathcal{L} = 2 S w - 2 \lambda w = \mathbf{0} \implies \mathbf{S w = \lambda w}$$

This is precisely the **Eigenvalue Equation**! Therefore, any stationary point $w$ must be an eigenvector of the covariance matrix $S$, with $\lambda$ as its corresponding eigenvalue.

#### 4. Maximizing Variance
Premultiplying by $w^T$:
$$\text{Var}(Xw) = w^T S w = w^T (\lambda w) = \lambda (w^T w) = \lambda$$

Thus, the variance along direction $w$ is identically equal to $\lambda$. To maximize variance, we must choose $\lambda = \lambda_{\max}$ (the **largest eigenvalue** of $S$) and $w = w_1$ (the corresponding normalized **eigenvector**). $\blacksquare$

### Question 2 (13%): Relationship with SVD and Why SVD is Preferred in Practice

#### 1. SVD Decomposition and Mathematical Connection
The Singular Value Decomposition of $X$ is $X = U \Sigma V^T$, where $U^T U = I_n$, $V^T V = I_p$, and $\Sigma = \text{diag}(\sigma_1, \dots, \sigma_p)$.

Computing the covariance matrix product $X^T X$:
$$X^T X = (U \Sigma V^T)^T (U \Sigma V^T) = V \Sigma^T U^T U \Sigma V^T = \mathbf{V \Sigma^2 V^T}$$

From this relation:
1. **Principal Loadings:** The right singular vectors $V$ are precisely the **eigenvectors of $X^T X$ (and $S$)**.
2. **Eigenvalues vs Singular Values:** The sample variance along principal component $i$ is $\lambda_i = \frac{\sigma_i^2}{n - 1}$.
3. **Principal Component Scores:** $Z = X V = (U \Sigma V^T) V = \mathbf{U \Sigma}$, obtaining coordinates directly without explicit matrix multiplication.

#### 2. Why SVD is Preferred over Eigendecomposition of $X^T X$
1. **Numerical Stability & Condition Number Squaring:**
 - Computing $X^T X$ squares the condition number: $\kappa(X^T X) = (\kappa(X))^2$.
 - In 64-bit IEEE floating-point arithmetic (~16 decimal digits), if $\kappa(X) = 10^4$, then $\kappa(X^TX) = 10^8$, losing half the available precision. Small singular values are destroyed by roundoff errors.
2. **Computational & Memory Efficiency:**
 - Forming $X^TX$ explicitly requires $O(n p^2)$ time and $O(p^2)$ memory.
 - Truncated/Randomized SVD can find the top $k$ principal components in $O(n p k)$ time without ever forming the dense $p \times p$ covariance matrix.

### Question 3 (7%): When Must Variables Be Standardised Before PCA? Financial Example

#### 1. General Rule
Standardization ($z = \frac{x - \mu}{\sigma}$) is mandatory whenever features are measured in **different units** or have **vastly different scales**.
Since PCA maximizes variance, unstandardized PCA is dominated purely by variables with large numerical magnitudes, regardless of information content.

#### 2. Corporate Financial Statement Example
Consider analyzing company financials with variables:
1. **Total Revenue / Assets (รายได้รวม):** Measured in Thai Baht (THB), values $\approx 10^{10} - 10^{11}$, variance $\approx 10^{20}$.
2. **Debt-to-Equity Ratio (D/E) (หนี้สินต่อทุน):** Dimensionless ratio, values $\approx 0.5 - 3.0$, variance $\approx 1$.
3. **Net Profit Margin (อัตรากำไรสุทธิ):** Percentage, values $\approx 0.05 - 0.20$, variance $\approx 0.01$.

- **Without Standardization:** Total Revenue variance ($10^{20}$) dwarfs D/E and Profit Margin by a factor of $10^{20}$. PC1 will simply measure company size, completely missing leverage, solvency, and profitability.
- **With Standardization:** All features have unit variance, allowing PCA to uncover true multidimensional patterns of financial health.

In [6]:
# สาธิตผลของสเกลตัวแปรต่อ PCA และเลือกจำนวนองค์ประกอบหลัก
# Part B: Empirical Verification of SVD vs Eigendecomposition
import numpy as np
import pandas as pd

np.random.seed(42)
n, p = 150, 4

# Simulate 4 financial features with vastly different scales:
# 1. Revenue (Billions)
# 2. D/E Ratio
# 3. Net Margin
# 4. Current Ratio
revenue = np.random.uniform(1e9, 5e10, size=(n, 1))
de_ratio = np.random.uniform(0.5, 3.5, size=(n, 1))
net_margin = np.random.uniform(-0.10, 0.25, size=(n, 1))
current_ratio = np.random.uniform(0.8, 2.5, size=(n, 1))

X_raw = np.hstack([revenue, de_ratio, net_margin, current_ratio])

# Standardize (Zero Mean, Unit Variance)
X_std = (X_raw - np.mean(X_raw, axis=0)) / np.std(X_raw, axis=0)

# Method 1: Via Covariance & Eigendecomposition
S = np.cov(X_std, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(S)
# Sort descending
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

# Method 2: Via SVD
U, Sigma, Vt = np.linalg.svd(X_std, full_matrices=False)
svd_var = (Sigma**2) / (n - 1)

print("=" * 70)
print("PCA VERIFICATION: Eigendecomposition vs SVD")
print("=" * 70)
comp_df = pd.DataFrame({
 "Component": [f"PC{i+1}" for i in range(p)],
 "Eigenvalue (Cov)": eigvals.round(4),
 "SVD Variance (Sigma^2 / (n-1))": svd_var.round(4),
 "Explained Variance Ratio": (eigvals / np.sum(eigvals)).round(4)
})
print(comp_df.to_string(index=False))

print("\nFirst Principal Direction Vector:")
print(f" Eigenvector PC1: {eigvecs[:, 0].round(4)}")
print(f" SVD V^T Row 1 : {Vt[0, :].round(4)} (Exact match up to sign flip)")
print("=" * 70)

PCA VERIFICATION: Eigendecomposition vs SVD
Component  Eigenvalue (Cov)  SVD Variance (Sigma^2 / (n-1))  Explained Variance Ratio
      PC1            1.1752                          1.1752                    0.2918
      PC2            1.0527                          1.0527                    0.2614
      PC3            0.9305                          0.9305                    0.2311
      PC4            0.8685                          0.8685                    0.2157

First Principal Direction Vector:
 Eigenvector PC1: [-0.6204 -0.3966  0.6096  0.2936]
 SVD V^T Row 1 : [ 0.6204  0.3966 -0.6096 -0.2936] (Exact match up to sign flip)
